In [ ]:
#| default_exp panels

In [ ]:
#| export
"""User-facing loading and inspection of existing panel stores."""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import pandas as pd

In [ ]:
#| export
@dataclass
class Panel:
    """An existing probe panel loaded without rewriting its stored values."""

    name: str
    candidates: pd.DataFrame
    source_path: Path

    def summary(self) -> pd.DataFrame:
        """Return a one-row overview using the historic score as stored."""
        data = self.candidates
        gene_column = "gene" if "gene" in data else "gene_name" if "gene_name" in data else None
        score_column = "raw_design_score" if "raw_design_score" in data else "score" if "score" in data else None
        return pd.DataFrame([{
            "panel": self.name,
            "probes": len(data),
            "genes": data[gene_column].nunique(dropna=True) if gene_column else 0,
            "median_score": data[score_column].median() if score_column else float("nan"),
            "score_column": score_column,
        }])

In [ ]:
#| export
def load_panel(path: str | Path, *, name: str | None = None) -> Panel:
    """Load a ProbeStore directory or its ``probes.parquet`` file.

    No normalization or rescoring is performed: historic and recalculated
    score columns remain separate and retain their stored values.
    """
    source_path = Path(path).expanduser()
    parquet_path = source_path / "probes.parquet" if source_path.is_dir() else source_path
    if not parquet_path.is_file():
        raise FileNotFoundError(f"Panel parquet file not found: {parquet_path}")
    candidates = pd.read_parquet(parquet_path)
    panel_name = name or parquet_path.parent.name
    return Panel(panel_name, candidates, parquet_path)